this is not really included in the project though , I suppose but yeah just if you also wanna understand how FFNN(ie Feed Forward Neural Nets work you can very well play along this or rather also use the dataset here too)

In [1]:
from NN import Linear, Sequential, ReLU, cross_entropy
import numpy as np

rng = np.random.default_rng(0)
n, K = 200, 3
X = np.zeros((n * K, 2)); y = np.zeros(n * K, dtype=int)
for k in range(K):
    r = np.linspace(0.05, 1, n); t = np.linspace(k * 4, (k + 1) * 4, n) + 0.2 * rng.standard_normal(n)
    X[k*n:(k+1)*n] = np.c_[r * np.sin(t), r * np.cos(t)]; y[k*n:(k+1)*n] = k
idx = rng.permutation(len(X)); cut = int(0.8 * len(X))
Xtr, ytr, Xva, yva = X[idx[:cut]], y[idx[:cut]], X[idx[cut:]], y[idx[cut:]]

model = Sequential(Linear(2, 64, rng=rng), ReLU(), Linear(64, 64, rng=rng), ReLU(),
                   Linear(64, K, rng=rng, init="xavier"))
params = model.parameters(); vel = [np.zeros_like(p.data) for p in params]
acc = lambda Xs, ys: (model(Xs).argmax(1) == ys).mean()
print(f"3-arm spiral  train={len(Xtr)} val={len(Xva)}")
for ep in range(1, 151):
    perm = rng.permutation(len(Xtr))
    for i in range(0, len(perm), 64):
        b = perm[i:i + 64]; model.zero_grad()
        _, dlogits = cross_entropy(model(Xtr[b]), ytr[b])
        model.backward(dlogits)
        for p, v in zip(params, vel):
            v *= 0.9; v += p.grad; p.data -= 0.05 * v
    if ep in (1, 37, 75, 150):
        print(f"epoch {ep:>3}  train acc {acc(Xtr, ytr):.3f}  val acc {acc(Xva, yva):.3f}")

3-arm spiral  train=480 val=120
epoch   1  train acc 0.515  val acc 0.583
epoch  37  train acc 1.000  val acc 0.992


epoch  75  train acc 1.000  val acc 0.992
epoch 150  train acc 1.000  val acc 0.992
